In [65]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from doubleml import DoubleMLData, DoubleMLIRM
from doubleml.datasets import make_irm_data
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [82]:
n, d = 4000, 10
X = np.random.normal(size=(n, d))
tau = 2 * (X[:, 0] > 0).astype(float) + 0.5 * X[:, 1]
mu0 = X[:, 0] + 0.5 * X[:, 1] + 0.1 * np.sin(X[:, 2])
pi = 1 / (1 + np.exp(-(X[:, 0] + X[:, 1])))

A = np.random.binomial(1, pi)
Y = mu0 + A * tau + np.random.normal(scale=1, size=n)

# train / test split for later visual check
X_tr, X_te, y_tr, y_te, a_tr, a_te, _, tau_true = train_test_split(
    X, Y, A, tau, test_size=0.30, random_state=1
)

In [71]:
X, Y, A = make_irm_data(n_obs=5000, dim_x=10, theta=1, return_type="array")
X_tr, X_te, y_tr, y_te, a_tr, a_te = train_test_split(
    X, Y, A, test_size=0.30, random_state=1
)

In [83]:
print("True treatment effect: ", tau_true.mean())

True treatment effect:  1.0172822401694586


In [84]:
class TARNet(nn.Module):
    def __init__(self, d, h=64, rep=2):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, rep), nn.ReLU()
        )
        self.f0 = nn.Linear(rep, 1)
        self.f1 = nn.Linear(rep, 1)

    def forward(self, x, a):
        phi = self.shared(x)
        y0, y1 = self.f0(phi), self.f1(phi)
        return torch.where(a.bool(), y1, y0), phi

net  = TARNet(d=10)
opt  = optim.Adam(net.parameters(), lr=1e-3)
loss = nn.MSELoss()

In [85]:
x_t = torch.tensor(X_tr, dtype=torch.float32)
x_test = torch.tensor(X_te, dtype=torch.float32)
a_t = torch.tensor(a_tr[:, None], dtype=torch.float32)
y_t = torch.tensor(y_tr[:, None], dtype=torch.float32)

In [86]:
for _ in range(800): 
    opt.zero_grad()
    y_hat, _ = net(x_t, a_t)
    loss(y_hat, y_t).backward()
    opt.step()

with torch.no_grad():
    phi_X = net.shared(torch.tensor(x_test, dtype=torch.float32)).numpy()

C:\Users\Work\AppData\Local\Temp\ipykernel_23300\2964891949.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  phi_X = net.shared(torch.tensor(x_test, dtype=torch.float32)).numpy()


In [87]:
def fit_irm(features):
    data = DoubleMLData.from_arrays(
        x = features,
        y = y_te,
        d = a_te
    )
    g = LGBMRegressor(verbose=-1, n_estimators=500, learning_rate=0.01)
    m = LGBMClassifier(verbose=-1, n_estimators=500, learning_rate=0.01)
    irm = DoubleMLIRM(data, ml_g=g, ml_m=m, n_folds=3, score='ATE')
    irm.fit()
    return irm

irm_X = fit_irm(X_te)
irm_P = fit_irm(phi_X)

c:\Users\Work\.conda\envs\dml_dev\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Work\.conda\envs\dml_dev\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Work\.conda\envs\dml_dev\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Work\.conda\envs\dml_dev\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Work\.conda\envs\dml_dev\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will b

In [88]:
print("IRM with X:")
print(irm_X)

IRM with X:
================== DoubleMLIRM Object ==================

------------------ Data summary      ------------------
Outcome variable: y
Treatment variable(s): ['d']
Covariates: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10']
Instrument variable(s): None
No. Observations: 1200

------------------ Score & algorithm ------------------
Score function: ATE

------------------ Machine learner   ------------------
Learner ml_g: LGBMRegressor(learning_rate=0.01, n_estimators=500, verbose=-1)
Learner ml_m: LGBMClassifier(learning_rate=0.01, n_estimators=500, verbose=-1)
Out-of-sample Performance:
Regression:
Learner ml_g0 RMSE: [[1.09235721]]
Learner ml_g1 RMSE: [[1.0923632]]
Classification:
Learner ml_m Log Loss: [[0.62245432]]

------------------ Resampling        ------------------
No. folds: 3
No. repeated sample splits: 1

------------------ Fit summary       ------------------
       coef   std err         t         P>|t|     2.5 %    97.5 %
d  0.915935  0.174475  

In [89]:
print("IRM with phi(X):")
print(irm_P)

IRM with phi(X):
================== DoubleMLIRM Object ==================

------------------ Data summary      ------------------
Outcome variable: y
Treatment variable(s): ['d']
Covariates: ['X1', 'X2']
Instrument variable(s): None
No. Observations: 1200

------------------ Score & algorithm ------------------
Score function: ATE

------------------ Machine learner   ------------------
Learner ml_g: LGBMRegressor(learning_rate=0.01, n_estimators=500, verbose=-1)
Learner ml_m: LGBMClassifier(learning_rate=0.01, n_estimators=500, verbose=-1)
Out-of-sample Performance:
Regression:
Learner ml_g0 RMSE: [[1.0645294]]
Learner ml_g1 RMSE: [[1.18244229]]
Classification:
Learner ml_m Log Loss: [[0.63697851]]

------------------ Resampling        ------------------
No. folds: 3
No. repeated sample splits: 1

------------------ Fit summary       ------------------
       coef   std err          t         P>|t|     2.5 %    97.5 %
d  1.076073  0.104667  10.280893  8.592743e-25  0.870929  1.281217

In [90]:
irm_X.confint()

,2.5 %,97.5 %
d,0.573971,1.257899


In [91]:
irm_P.confint()

,2.5 %,97.5 %
d,0.870929,1.281217
